# Готовая модель движения сердца MONAI Physio

**Вопрос исследования.** Улучшает ли готовая модель движения совпадение формы сердца с реально снятыми фазами 4D-КТ Ника, если индивидуальная подгонка выполнена только по фазе 00? В опыте использованы исторические сердечные веса MONAI Physio/PhysicsNeMo без обучения и дообучения на данных проекта. Проверка завершена 1 октября 2026 года; подтверждения прогностической точности пока нет.

Предсказание этой модели оценивается отдельно от качества самой автоматической сегментации. Результат сети — изменение поверхности, а проверочный объект — независимо вычисленная TotalSegmentator v2 поверхность соответствующей КТ-фазы.

## Модель и индивидуальный вход

В установленной версии MONAI Physio 2026.09.1 доступен интерфейс сетевого вывода. Для сердца взяты готовые веса из официального исторического выпуска 2026.07.3, а не текущие готовые веса движения лёгкого. Сопутствующий шаблон содержит 167 240 вершин и использует 15 коэффициентов PCA; метаданные checkpoint перечисляют 28 обучающих случаев с обозначениями `pm*`. Файл PCA полностью получен и проверен по SHA-256. Техническое сопоставление старых имён полей с текущим интерфейсом не меняло сетевые веса.

Индивидуальные входы ограничены 3D-КТ Ника и автоматической поверхностью всего сердца из её фазы 00. Сначала к этой поверхности подогнаны положение, линейное преобразование и 15 коэффициентов формы исторического шаблона. Автоматическая маска фазы 00 используется при подгонке и не является независимым контролем её точности. Маски остальных фаз не участвовали в этой операции. Исходная поверхность Duke Heart и класс `whole_heart` TotalSegmentator v2 могут охватывать разные анатомические области; их эквивалентность не установлена.

## Протокол прогноза и сравнения

Сеть получает коэффициенты формы и долю сердечного цикла. Доля для фазы 00 равна 0,14, для фаз 01–08 — соответственно 0,23; 0,33; 0,43; 0,53; 0,62; 0,72; 0,82; 0,92. Эти числа взяты из DICOM-тега `NominalPercentageOfCardiacPhase`. Они задают номинальные точки R–R, но не являются независимыми метками механического сокращения. Фазы с долей не менее 1 не подавались в модель, работающую на шкале от 0 до 1.

Для каждой целевой доли рассчитано сетевое смещение; из него вычтено смещение, рассчитанное для 0,14. Полученная разность перенесена на форму, подогнанную к фазе 00. **Контрольное предсказание** — та же подогнанная форма без движения. После фиксации прогноза каждую из этих двух поверхностей сравнили с автоматической поверхностью реально снятой целевой фазы.

Основная метрика — симметричная средняя дистанция ближайших **вершин** двух поверхностей, в миллиметрах. Функция `distances()` ищет ближайшие вершины через KDTree в каждом направлении; она не ищет ближайшую точку внутри треугольника. Меньшее значение означает лучшее совпадение по этой метрике. Результат зависит от плотности и распределения вершин, не является точной дистанцией «точка — треугольная поверхность» (ASSD), не прослеживает соответствующие анатомические точки и не измеряет истинное поле деформации. Первичный оперативный контроль прореживал вершины с шагом 16; приведённые далее итоговые числа пересчитаны по **всем вершинам** сохранённых поверхностей без новой подгонки.

## Качество исходной подгонки

Среднее симметричное расстояние от подогнанного шаблона до автоматической поверхности фазы 00 составило **4,384 мм** при расчёте по всем вершинам. В оперативном прореженном контроле аффинная подгонка дала 5,093 мм, а добавление PCA снизило показатель до 4,935 мм. Эти прореженные числа нельзя напрямую вычитать из полной оценки 4,384 мм: наборы оцениваемых вершин различаются.

Аффинная часть имела определитель 1,041 и сингулярные значения 1,051; 1,032; 0,959, что исключает грубое схлопывание именно этого преобразования. Несколько миллиметров остаточного расхождения и различие анатомических определений ограничивают смысл последующего прогноза: даже правильно вычисленные сетевые смещения могут относиться не к той границе сердца, которую мы проверяем.

## Прогноз на удержанных фазах Ника

| Фаза | Доля R–R | Неподвижная форма, мм | Прогноз, мм | Прогноз минус контроль, мм | Среднее сетевое смещение, мм |
|---|---:|---:|---:|---:|---:|
| 01 | 0,23 | 4,091 | 4,308 | +0,216 | 3,069 |
| 02 | 0,33 | 4,521 | 4,459 | −0,063 | 3,283 |
| 03 | 0,43 | 4,499 | 4,574 | +0,075 | 1,828 |
| 04 | 0,53 | 4,314 | 4,531 | +0,217 | 2,634 |
| 05 | 0,62 | 4,352 | 4,475 | +0,123 | 2,811 |
| 06 | 0,72 | 4,540 | 4,347 | −0,193 | 3,114 |
| 07 | 0,82 | 4,633 | 4,410 | −0,222 | 3,963 |
| 08 | 0,92 | 5,122 | 4,758 | −0,364 | 4,337 |

Отрицательная разность означает улучшение прогноза по выбранной метрике. Оно наблюдалось на фазах 02, 06, 07 и 08; на фазах 01, 03, 04 и 05 совпадение ухудшилось. Среднее по восьми фазам составило **4,509 мм для неподвижной формы и 4,483 мм для прогноза**, то есть улучшение всего 0,026 мм, или около 0,58 %. Целевые маски независимы от сетевого вывода, но их анатомическая точность не проверена вручную.

## Научная оценка и применимость

Готовая сеть технически выполняет вывод по одной КТ и заданной номинальной доле R–R. В данном опыте движение величиной несколько миллиметров **не дало устойчивого выигрыша** над неподвижной подогнанной формой: четыре фазы улучшились и четыре ухудшились. Остаточная ошибка подгонки исходной формы намного больше среднего выигрыша. По этим данным сетевой вывод нельзя признать достоверной персональной кинематикой или передавать как проверенную движущуюся геометрию в FEM.

Для пересмотра решения нужна готовая модель с анатомически согласованным шаблоном и отдельная проверка фазовых контуров; затем потребуется повторить сравнение с тем же неподвижным контролем и проверить сетку и электрический ответ при одной FEM-постановке. Пользовательский протокол и численные агрегаты приведены в [отчёте о прогнозе](mp06_nix_one_ct_forecast_report.md), реализация — в [скрипте](mp06_nix_one_ct_forecast.py). Исторический технический запуск на средней форме описан в [JSON](mp06_legacy_checkpoint_smoke.json): он подтверждает только работоспособность весов. Индивидуальные поверхности, веса и полный протокол прогона хранятся локально вне Git.

## Интерактивный просмотр рассчитанного движения

Ниже показаны девять поверхностей Ника, полученных готовой сердечной сетью по
одной фазе 00. Сцену можно вращать и увеличивать, фазу выбирать ползунком или
воспроизводить последовательность кадров. Оси заданы в LPS, мм. Номинальные
доли R–R — вход модели; скорость воспроизведения не измерялась. Для вывода
сетка упрощена одинаковым способом во всех кадрах; показатели точности в
отчёте рассчитывались по полным поверхностям. Эта визуализация показывает
результат сети, но не подтверждает его анатомическую точность. Ячейка требует
локальные исключённые из Git поверхности в `runs/mp06_nix_one_ct_forecast/`.


In [ ]:
from pathlib import Path
from IPython.display import HTML, display
from mp_visualize_motion import motion_figure

surface_dir = Path("runs/mp06_nix_one_ct_forecast")
phases = [f"{number:02d}" for number in range(9)]
frames = [
    (phase, {"Сетевой прогноз сердца": surface_dir / f"phase_{phase}_predicted_surface.vtp"})
    for phase in phases
]
figure = motion_figure(
    frames,
    title="Ник: готовый сетевой прогноз формы сердца",
    subtitle="Фазы 00–08, номинальная доля R–R; кадры — прогноз из фазы 00, не измерение движения.",
    colors={"Сетевой прогноз сердца": "#d75864"},
    opacity={"Сетевой прогноз сердца": 0.96},
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))
# Этот локальный HTML содержит индивидуальную геометрию и исключён из Git.
output = surface_dir / "heart_forecast_interactive_3d.html"
figure.write_html(str(output), include_plotlyjs=True)
print("Локальный интерактивный HTML:", output)